# NPS Classifier: Model Selection and Validation

**Context:** This notebook focuses on the training, validation, and comparison of machine learning models for the classification of New Psychoactive Substances (NPS) based on Electron Ionization (EI) Mass Spectra.

**Objectives:**
1.  **Feature Engineering:** Extraction of statistical and spectral features from raw mass spectra.
2.  **Model Training:** Training of three distinct architectures:
    * **Balanced Random Forest (BRF):** To handle class imbalance.
    * **Deep Neural Network (DNN):** A dense neural network for spectral pattern recognition.
    * **XGBoost:** Gradient boosting decision trees (Selected as the *Champion Model* for PENTION-M).
3.  **Benchmarking:** Comparison against legacy models (Baseline/Claudio's implementation) to quantify performance improvements.

# IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

# Scikit-Learn & Imbalanced Learn
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, cross_val_score, 
                                     StratifiedKFold, RandomizedSearchCV)
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, roc_curve, auc)
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.feature_selection import RFE
from sklearn.tree import plot_tree

# XGBoost
from xgboost import XGBClassifier

# TensorFlow / Keras
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import plot_model

# Configure plotting style
sns.set_style("whitegrid")

# Dataset Loading & EDA

In [ ]:
# Loading the complete EI spectrum dataset
nps_dataset = pd.read_csv('datasetNPS/PENTION_EI_Complete.csv', sep=',', header=0)

print(f'[INFO] The dataset has {nps_dataset.shape[0]} instances and {nps_dataset.shape[1]} features.')

# Quick inspection
# nps_dataset.info()
# print(nps_dataset.isnull().sum(axis=1))

# --- CLASS DISTRIBUTION ANALYSIS ---
classes = ['Cathinone analogues', 'Cannabinoid analogues',
           'Phenethylamine analogues', 'Piperazine analogues',
           'Tryptamine analogues', 'Fentanyl analogues', 'Other compounds']

legends = {
  0: 'Cathinone analogues',
  1: 'Cannabinoid analogues',
  2: 'Phenethylamine analogues',
  3: 'Piperazine analogues',
  4: 'Tryptamine analogues',
  5: 'Fentanyl analogues',
  6: 'Other compounds'
}

distributions = nps_dataset['label'].value_counts().sort_index()

plt.figure(figsize=(10, 6))
ax = distributions.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Distribution of Classes in the Dataset', fontsize=14)
plt.xlabel('Substance Classes', fontsize=12)
plt.ylabel('Number of Instances', fontsize=12)
plt.xticks(ticks=range(len(classes)), labels=[legends[i] for i in range(len(classes))], rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add count annotations
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), 
        (p.get_x() + p.get_width() / 2, p.get_height()), 
        ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

# --- SPECTRUM VISUALIZATION SAMPLE ---
# Visualizing a specific compound to verify spectral data integrity
compound_name = "JWH-116"
row = nps_dataset[nps_dataset.iloc[:, 0] == compound_name].iloc[0]
spectrum = row[1:-1].astype(float).values 
mz = list(range(1, 601))

plt.figure(figsize=(14, 5))
(markerline, stemlines, baseline) = plt.stem(mz, spectrum)
plt.setp(stemlines, 'linewidth', 1)
plt.xlabel("m/z (Mass-to-Charge Ratio)")
plt.ylabel("Relative Intensity")
plt.title(f"Mass Spectrum: {compound_name}")
plt.grid(True, linestyle=':', alpha=0.6)

# Annotate peaks above a certain threshold for clarity
threshold = 10.0
for mzi, intensity in zip(mz, spectrum):
    if intensity > threshold:
        plt.text(mzi, intensity + 1, f"{intensity:.1f}", ha='center', va='bottom', fontsize=8, rotation=90)

plt.tight_layout()
plt.show()

# Feature Engineering

To enhance the discriminative power of traditional machine learning models (like Random Forest), we extract domain-specific chemical features from the raw spectra.

| Feature Name | Description |
|--------------|-------------|
| **Base Peak Mass** | m/z of the peak with the highest intensity (100% relative abundance). |
| **Base Peak Mass Proximity** | m/z difference between the base peak and its nearest neighbor. |
| **Maximum Mass** | The highest m/z value observed in the spectrum (often related to molecular ion). |
| **Maximum Mass Proximity** | Difference between the max mass peak and its nearest neighbor. |
| **Number of Peaks** | Total count of peaks with non-zero intensity. |
| **Intensity Stats** | Mean, Std Dev, and Density of intensity values. |
| **Mass Stats** | Mean, Std Dev, and Density of m/z values. |
| **PPMD** | (Peakwise Mass Difference) Most frequent difference between peak pairs (fragmentation patterns). |

In [ ]:
# --- FEATURE EXTRACTION LOGIC ---

mz_range = np.arange(1, 601)
# Extract only spectral columns for processing
spectra_values = nps_dataset[mz_range.astype(str)].values

def compute_features(spectrum):
    """
    Computes statistical and chemical features from a single mass spectrum.
    """
    peaks = [(mz, intensity) for mz, intensity in zip(mz_range, spectrum) if intensity > 0]
    
    if not peaks:
        return [np.nan] * 13

    mz_values, intensities = zip(*peaks)
    mz_values = np.array(mz_values)
    intensities = np.array(intensities)
    
    # 1. Base Peak Metrics
    base_peak_idx = np.argmax(intensities)
    base_peak_mass = mz_values[base_peak_idx] 
    
    if len(mz_values) > 1:
        mz_diff_base = np.abs(mz_values - base_peak_mass)
        # Find nearest neighbor distance (excluding self, which is 0)
        base_prox = np.partition(mz_diff_base[mz_diff_base != 0], 0)[0]
    else:
        base_prox = 0.0
        
    # 2. Maximum Mass Metrics
    max_mass = np.max(mz_values)
    if len(mz_values) > 1:
        mz_diff_max = np.abs(mz_values - max_mass)
        max_prox = np.partition(mz_diff_max[mz_diff_max != 0], 0)[0]
    else:
        max_prox = 0.0
        
    # 3. Statistical Metrics
    num_peaks = len(peaks)
    intensity_mean = np.mean(intensities)
    intensity_std = np.std(intensities)
    intensity_density = np.max(intensities) / num_peaks
    mass_mean = np.mean(mz_values)
    mass_std = np.std(mz_values)
    mass_density = max_mass / num_peaks
    
    # 4. Fragmentation Patterns (PPMD)
    diffs = np.abs(np.subtract.outer(mz_values, mz_values))
    # Take upper triangle to avoid duplicates and self-differences
    diffs = diffs[np.triu_indices(len(diffs), k=1)]
    
    if len(diffs) > 0:
        diff_counts = np.bincount(np.round(diffs).astype(int))
        ppmd = np.argmax(diff_counts)
        mean_ppmd = np.mean(diffs)
    else:
        ppmd = 0
        mean_ppmd = 0.0

    return [
        base_peak_mass, base_prox, max_mass, max_prox,
        num_peaks, intensity_mean, intensity_std, intensity_density,
        mass_mean, mass_std, mass_density, ppmd, mean_ppmd
    ]

# Apply extraction
features = np.array([compute_features(s) for s in spectra_values])

feature_columns = [
    "BasePeakMass", "BasePeakMassProximity", "MaxMass", "MaxMassProximity",
    "NumPeaks", "IntensityMean", "IntensityStd", "IntensityDensity",
    "MassMean", "MassStd", "MassDensity", "PPMD", "MeanPPMD"
]

features_df = pd.DataFrame(features, columns=feature_columns)
full_df = pd.concat([nps_dataset[["Name", "label"]], features_df], axis=1)

print("[INFO] Feature extraction complete.")
full_df.head()

# EXPLORATORY ANALYSIS: CORRELATION

In [ ]:
# Drop Name column for analysis
analysis_df = full_df.drop(columns=['Name'])

plt.figure(figsize=(12, 10))
sns.heatmap(analysis_df.corr(), annot=True, fmt=".2f", cmap='coolwarm', square=True, cbar_kws={"shrink": .8})
plt.title('Correlation Matrix of Engineered Features')
plt.tight_layout()
plt.show()

# --- FEATURE SELECTION (Collinearity Reduction) ---
# Removing highly correlated features to reduce redundancy
X_temp = full_df.drop(columns=['label', 'Name'])
y_temp = full_df['label']

# Fit a temp RF to judge importance vs correlation
rf_temp = RandomForestClassifier(random_state=42)
rf_temp.fit(X_temp, y_temp)
importances_temp = pd.Series(rf_temp.feature_importances_, index=X_temp.columns)

corr_matrix = X_temp.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

correlation_threshold = 0.80
to_drop = set()

for col in upper.columns:
    correlated_features = upper.index[upper[col] > correlation_threshold].tolist()
    for corr_feat in correlated_features:
        # If correlated, drop the one with lower importance
        if importances_temp[col] < importances_temp[corr_feat]:
            to_drop.add(col)
        else:
            to_drop.add(corr_feat)

print(f"[INFO] Features removed due to high correlation (> {correlation_threshold}) and low importance:", to_drop)

# Update dataset
full_df = full_df.drop(columns=list(to_drop))
feature_columns = full_df.drop(columns=['label', 'Name']).columns.tolist()

# Visual check after removal
plt.figure(figsize=(10, 8))
sns.heatmap(full_df.drop(columns=['Name']).corr(), annot=True, fmt=".2f", cmap='coolwarm', square=True, cbar_kws={"shrink": .8})
plt.title('Correlation Matrix (Reduced Feature Set)')
plt.tight_layout()
plt.show()

# MODEL 1: BALANCED RANDOM FOREST (BRF)

In [ ]:
# Data Preparation
df_model = full_df.copy()  
Y = df_model['label']
X = df_model.drop(['label', 'Name'], axis=1)
X = np.array(X)
Y = np.array(Y)

print(f"Input Shape: {X.shape}, Target Shape: {Y.shape}")

# Stratified Split
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

# SMOTE Oversampling
smote = SMOTE(random_state=42)
x_train, y_train = smote.fit_resample(x_train, y_train)
print("Training Class Distribution after SMOTE:\n", pd.Series(y_train).value_counts())

# BRF Training
clf_brf = BalancedRandomForestClassifier(n_estimators=100, random_state=42)
clf_brf.fit(x_train, y_train)

# Evaluation
y_pred = clf_brf.predict(x_test)
class_labels = [legends[i] for i in sorted(np.unique(Y))]

print("\n--- CLASSIFICATION REPORT (Initial BRF) ---")
print(classification_report(y_test, y_pred, target_names=class_labels))

# Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('Confusion Matrix - BRF')
plt.xlabel('Predicted Class')
plt.ylabel('Actual Class')
plt.show()

# Cross Validation
scores = cross_val_score(clf_brf, x_train, y_train, cv=4)
print(f"\nCross-validation accuracy (4-fold): {scores.mean():.2f} ± {scores.std():.2f}")

# Save Model
joblib.dump(clf_brf, 'model/balanced_random_forest_brf.pkl')

# --- ROC CURVE (BRF) ---
classes_unique = np.unique(y_test)
n_classes = len(classes_unique)
y_test_bin = label_binarize(y_test, classes=classes_unique)
y_score = clf_brf.predict_proba(x_test)

fpr = dict()
tpr = dict()
roc_auc = dict()

plt.figure(figsize=(10, 8))
colors = plt.cm.get_cmap('tab10', n_classes)

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    plt.plot(fpr[i], tpr[i], color=colors(i), lw=2,
             label=f"{class_labels[i]} (AUC = {roc_auc[i]:.2f})")

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Balanced Random Forest')
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

# RECURSIVE FEATURE ELIMINATION (RFE)

In [ ]:
# --- RECURSIVE FEATURE ELIMINATION (RFE) ---

# RFE Setup
rfe_base_clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf_rfe = RFE(rfe_base_clf, n_features_to_select=5, verbose=1)
clf_rfe.fit(x_train, y_train)

# Visualize Feature Importances from RFE
importances_rfe = clf_rfe.estimator_.feature_importances_
indices = np.argsort(importances_rfe)[::-1]

plt.figure(figsize=(12, 6))
plt.title("Feature Importances from RFE Step")
plt.bar(range(len(importances_rfe)), importances_rfe[indices], color='teal', align='center')
plt.xticks(range(len(importances_rfe)), [feature_columns[i] for i in indices], rotation=45, ha='right')
plt.xlim([-1, len(importances_rfe)])
plt.xlabel("Features")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

# Select Features
selected_indices = clf_rfe.support_
selected_features = np.array(feature_columns)[selected_indices]
print("[INFO] Selected Features:", selected_features)

# Reduce Dataset
X_train_selected = x_train[:, selected_indices]
X_test_selected = x_test[:, selected_indices]

# Retrain BRF on Selected Features
bdt_clf_rfe = BalancedRandomForestClassifier(n_estimators=100, random_state=42)
bdt_clf_rfe.fit(X_train_selected, y_train)
y_pred_rfe = bdt_clf_rfe.predict(X_test_selected)

print("\n--- CLASSIFICATION REPORT (After RFE) ---")
print(classification_report(y_test, y_pred_rfe, target_names=class_labels))

# Confusion Matrix RFE
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_test, y_pred_rfe), annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('Confusion Matrix - RFE')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Cross Validation RFE
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
scores = cross_val_score(bdt_clf_rfe, X_train_selected, y_train, cv=cv)
print(f"\nCross-validation accuracy (4-fold) after RFE: {scores.mean():.2f} ± {scores.std():.2f}")

joblib.dump(bdt_clf_rfe, 'model/balanced_random_forest_rfe.pkl')

# --- ROC CURVE (RFE) ---
y_score_rfe = bdt_clf_rfe.predict_proba(X_test_selected)
fpr, tpr, roc_auc = {}, {}, {}

plt.figure(figsize=(10, 8))
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score_rfe[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    plt.plot(fpr[i], tpr[i], color=colors(i), lw=2,
             label=f"{class_labels[i]} (AUC = {roc_auc[i]:.2f})")

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Balanced Random Forest (RFE)')
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

# HYPERPARAMETER TUNING (RandomizedSearchCV)

In [ ]:
param_distributions = {
    'n_estimators': [10, 50, 100, 150],
    'max_depth': [None, 5, 10, 20, 50],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

random_search = RandomizedSearchCV(
    estimator=bdt_clf_rfe,
    param_distributions=param_distributions,
    n_iter=20, cv=3, scoring='accuracy', n_jobs=-1, random_state=42
)

random_search.fit(x_train, y_train)
print("Best parameters found: ", random_search.best_params_)
print("Best cross-validation score: ", random_search.best_score_)

best_clf = random_search.best_estimator_
y_pred_best = best_clf.predict(x_test)

print("\n--- CLASSIFICATION REPORT (Best Model) ---")
print(classification_report(y_test, y_pred_best, target_names=class_labels))

# Confusion Matrix Best Model
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_test, y_pred_best), annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('Confusion Matrix - Best Model')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Cross Validation Best Model
scores_best = cross_val_score(best_clf, x_test, y_test, cv=4)
print(f"Cross-validation accuracy (4-fold): {scores_best.mean():.2f} ± {scores_best.std():.2f}")

# --- TREE ANALYSIS ---
# Visualization of a single tree
plt.figure(figsize=(20, 10))
plot_tree(
    best_clf.estimators_[0],
    feature_names=selected_features,
    class_names=[legends[i] for i in best_clf.classes_],
    filled=True, rounded=True,
)
plt.title("Visualizing a Single Tree in Balanced Random Forest")
plt.tight_layout()
plt.show()

# Average Tree Metrics
print("\n--- AVERAGE TREE METRICS ---")
metrics = {
    'n_nodes': [est.tree_.node_count for est in best_clf.estimators_],
    'max_depth': [est.tree_.max_depth for est in best_clf.estimators_]
}
metrics_df = pd.DataFrame(metrics)
print(metrics_df.mean())

metrics_df.mean().plot(kind='bar', figsize=(8, 5), color='teal')
plt.title('Average Tree Metrics - Balanced Random Forest')
plt.ylabel('Average Value')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# --- ROC CURVE (Best Model) ---
y_score = best_clf.predict_proba(x_test)
fpr, tpr, roc_auc = {}, {}, {}

plt.figure(figsize=(10, 8))
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    plt.plot(fpr[i], tpr[i], color=colors(i), lw=2,
             label=f" {class_labels[i]} (AUC = {roc_auc[i]:.2f})")

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Balanced Random Forest (Best Model)')
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

joblib.dump(best_clf, 'model/balanced_random_forest_brf_hmp.pkl')
x_test_brf = X_test_selected.copy()

# MODEL 2: DEEP NEURAL NETWORK (DNN)

In [ ]:
# Prepare dataset (using raw spectra, not engineered features)
spectrum_dataset = nps_dataset.copy().drop(columns=['Name'])
print(f'[INFO] The spectrum dataset has {spectrum_dataset.shape[0]} instances and {spectrum_dataset.shape[1]} features.')

# Model Architecture
model = keras.Sequential(name='dnn')
model.add(keras.Input(shape=(600,)))
model.add(layers.Dense(300, activation='relu'))
model.add(layers.Dropout(0.2))
model.add(layers.Dense(30, activation='relu'))
model.add(layers.Dropout(0.2))
model.add(layers.Dense(7, activation='softmax'))

print(model.summary())
plot_model(model, to_file="dnn_model_structure.png", show_shapes=True, show_layer_names=True, dpi=96)

# Compile
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy'],
)

# Data Split & Scaling
Y = spectrum_dataset['label']
X = spectrum_dataset.drop(['label'], axis=1)
X = np.array(X)
Y = np.array(Y)

print(f"Shapes: X={X.shape}, Y={Y.shape}")
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# Oversampling
print("Class dist before SMOTE:", pd.Series(y_train).value_counts())
smote = SMOTE(random_state=42)
x_train, y_train = smote.fit_resample(x_train, y_train)
print("Class dist after SMOTE:", pd.Series(y_train).value_counts())

# Training
early_stop = EarlyStopping(
    monitor='val_loss',     
    patience=5,             
    restore_best_weights=True 
)

history = model.fit(
    x_train, y_train,  
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop] 
)

# Training Curves
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

# Evaluation
y_pred_dnn = model.predict(x_test)
y_pred_classes = np.argmax(y_pred_dnn, axis=1)

print("\n--- CLASSIFICATION REPORT (DNN) ---")
print(classification_report(y_test, y_pred_classes, target_names=class_labels))

# Confusion Matrix
confusion_matrix_result = confusion_matrix(y_test, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix_result, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('Confusion Matrix - DNN')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.grid(False)
plt.show()

# ROC Curve DNN
y_score_dnn = model.predict(x_test) # Softmax probabilities
classes = np.unique(y_test)
n_classes = len(classes)
y_test_bin = label_binarize(y_test, classes=classes)

fpr, tpr, roc_auc = {}, {}, {}
plt.figure(figsize=(8, 6))
colors = plt.cm.get_cmap('tab10', n_classes)

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score_dnn[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    plt.plot(fpr[i], tpr[i], color=colors(i), lw=2,
             label=f" {class_labels[i]} (AUC = {roc_auc[i]:.2f})")

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Deep Neural Network')
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

model.save('model/dnn_spectra_version.keras')
joblib.dump(scaler, "model/scale_dnn.pkl")

# MODEL 3: XGBOOST (CHAMPION MODEL)

In [ ]:
# Loading Model & Scaler
scaler_xgb = joblib.load("model/xgb_scaler.pkl")
xgb_model = XGBClassifier()
xgb_model.load_model("model/xgb_nps_model.json")

# Data Prep for XGBoost
X_spectrum = spectrum_dataset.drop(columns=["label"]).values
Y_spectrum = spectrum_dataset["label"].values
x_train_s, x_test_s, y_train_s, y_test_s = train_test_split(
    X_spectrum, Y_spectrum, test_size=0.2, random_state=42, stratify=Y_spectrum
)

x_test_s_scaled = scaler_xgb.transform(x_test_s)
y_pred_xgb = xgb_model.predict(x_test_s_scaled)

# 1) Accuracy & Report
acc_xgb = accuracy_score(y_test_s, y_pred_xgb)
print(f"\nAccuracy XGBoost: {acc_xgb:.4f}")

print("\n--- CLASSIFICATION REPORT (XGBoost) ---")
print(classification_report(y_test_s, y_pred_xgb, target_names=class_labels))

# 2) Confusion Matrix
cm = confusion_matrix(y_test_s, y_pred_xgb)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title("XGBoost – Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

# 3) Feature Importance
importances = xgb_model.feature_importances_
plt.figure(figsize=(14, 6))
plt.bar(range(len(importances)), importances)
plt.title("XGBoost – Feature Importance (Global)")
plt.xlabel("Feature index (m/z)")
plt.ylabel("Importance")
plt.show()

# 4) ROC Curve Multiclass
y_test_bin = label_binarize(y_test_s, classes=np.unique(Y_spectrum))
y_score = xgb_model.predict_proba(x_test_s_scaled)

fpr, tpr, roc_auc = {}, {}, {}
plt.figure(figsize=(10, 8))
colors = plt.colormaps.get_cmap("tab10")

for i in range(len(class_labels)):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    plt.plot(fpr[i], tpr[i], lw=2, color=colors(i),
             label=f"{class_labels[i]} (AUC={roc_auc[i]:.2f})")

plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("XGBoost – ROC Curve (Multiclass)")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

# 5) Cross Validation (Training set only)
print("\nRunning Cross-Validation (5-fold)...")
x_train_scaled = scaler_xgb.transform(x_train_s)
cv_scores = cross_val_score(
    xgb_model, x_train_scaled, y_train_s,
    cv=5, scoring='accuracy'
)
print(f"Cross-val mean accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# COMPARISON WITH CLAUDIO'S ORIGINAL MODELS

In [ ]:
# 1. Balanced Random Forest Comparison
try:
    brf_original = joblib.load("model/Claudio/balanced_random_forest_brf.pkl")
    # Handle feature mismatch
    if hasattr(brf_original, "n_features_in_"):
        n_features_old = brf_original.n_features_in_
        print(f"[INFO] The original BRF model expects {n_features_old} features. Using the first {n_features_old}.")
        # Note: Accessing x_test from cell 10 might be wrong dimension if BRF used extracted features. 
        # Using x_test (extracted features) from Cell 7 logic, which corresponds to 'x_test' variable here unless overwritten.
        # Assuming x_test still holds feature-engineered data from earlier cells:
        # However, variable name reuse in Cell 10 (DNN) overwrote x_test with spectra.
        # Recovering from x_test_brf backup:
        x_test_brf_old = x_test_brf[:, :n_features_old]
    else:
        x_test_brf_old = x_test_brf

    print("\n\nBalancedRandomForest comparison: New vs. Original (Claudio)")
    y_pred_new = best_clf.predict(x_test_brf)
    y_pred_old = brf_original.predict(x_test_brf_old)
    
    acc_new = accuracy_score(y_test, y_pred_new) # y_test is shared (labels same)
    acc_old = accuracy_score(y_test, y_pred_old)
    
    print(f"\nAccuracy (New Model): {acc_new:.4f}")
    print(f"Accuracy (Legacy Model - Claudio): {acc_old:.4f}")
    
    # Confusion Matrix Legacy
    confusion_matrix_result = confusion_matrix(y_test, y_pred_old)
    plt.figure(figsize=(10, 8))
    sns.heatmap(confusion_matrix_result, annot=True, fmt='d', cmap='Oranges',
                xticklabels=class_labels, yticklabels=class_labels)
    plt.title('Confusion Matrix - BRF Claudio')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.grid(False)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"[WARNING] Could not compare BRF: {e}")
    acc_old = 0.0

# 2. DNN Comparison
try:
    print("\n\nDNN Comparison: New vs. Original (Claudio)")
    dnn_original = load_model("model/Claudio/dnn_spectra_version.keras")
    
    # x_test here is from Cell 10 (Spectra) which is correct for DNN
    y_pred_dnn_new = model.predict(x_test)
    y_pred_dnn_old = dnn_original.predict(x_test)
    
    y_classes_new = np.argmax(y_pred_dnn_new, axis=1)
    y_classes_old = np.argmax(y_pred_dnn_old, axis=1)
    
    acc_new_dnn = accuracy_score(y_test, y_classes_new)
    acc_old_dnn = accuracy_score(y_test, y_classes_old)
    
    print(f"\nAccuracy (New DNN): {acc_new_dnn:.4f}")
    print(f"Accuracy (Legacy Model - Claudio): {acc_old_dnn:.4f}")
    
    print("\nClassification Report (Legacy DNN)")
    print(classification_report(y_test, y_classes_old, target_names=class_labels))
    
    # Confusion Matrix Legacy
    confusion_matrix_result = confusion_matrix(y_test, y_classes_old)
    plt.figure(figsize=(10, 8))
    sns.heatmap(confusion_matrix_result, annot=True, fmt='d', cmap='Purples',
                xticklabels=class_labels, yticklabels=class_labels)
    plt.title('Confusion Matrix - DNN Claudio')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.grid(False)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"[WARNING] Could not compare DNN: {e}")

# FINAL COMPARISON
print("FINAL MODEL COMPARISON")

print(f"\nAccuracy BRF (Claudio): {acc_old:.4f}")
print(f"Accuracy BRF New (Best Model): {acc_new:.4f}")
print(f"Final XGBoost Accuracy (PENTION-M): {acc_xgb:.4f}")

print("\nBEST MODEL:", 
      ["BRF Claudio", "BRF New", "XGBoost"][np.argmax([
          acc_old,
          acc_new,
          acc_xgb
      ])])